# Data Cleaning & Feature Engineering

This notebook prepares VCT 2025 match data for two downstream goals: descriptive stats on agent pick rate and win rate, and predictive models estimating win probability from team role composition. It outputs three processed files: `agent_level.csv`, `comp_only.csv`, and `comp_map.csv`.

In [1]:
import pandas as pd
import numpy as np
import glob

In [2]:
files = glob.glob('../data/raw/**/detailed_matches_player_stats.csv', recursive=True)
df_raw = pd.concat([pd.read_csv(file) for file in files], ignore_index=True)
df = df_raw.copy()

In [3]:
df.head()

,match_id,event_name,event_stage,match_date,team1,team2,score_overall,player_name,player_id,player_team,...,a,kd_diff,kast,adr,hs_percent,fk,fd,fk_fd_diff,map_name,map_winner
0,542195,Valorant Champions 2025,Group Stage: \n\t\t\t\t\t\tOpening (A),2025-09-12 09:00:00,Paper Rex,Xi Lai Gaming,2 - 0,something,17086.0,Paper Rex,...,21.0,17.0,88%,188.0,20%,6.0,1.0,5.0,NaN,NaN
1,542195,Valorant Champions 2025,Group Stage: \n\t\t\t\t\t\tOpening (A),2025-09-12 09:00:00,Paper Rex,Xi Lai Gaming,2 - 0,f0rsakeN,9801.0,Paper Rex,...,17.0,16.0,78%,165.0,37%,6.0,7.0,-1.0,NaN,NaN
2,542195,Valorant Champions 2025,Group Stage: \n\t\t\t\t\t\tOpening (A),2025-09-12 09:00:00,Paper Rex,Xi Lai Gaming,2 - 0,Jinggg,7378.0,Paper Rex,...,12.0,12.0,78%,183.0,16%,6.0,5.0,1.0,NaN,NaN
3,542195,Valorant Champions 2025,Group Stage: \n\t\t\t\t\t\tOpening (A),2025-09-12 09:00:00,Paper Rex,Xi Lai Gaming,2 - 0,d4v41,9803.0,Paper Rex,...,12.0,0.0,73%,107.0,22%,2.0,3.0,-1.0,NaN,NaN
4,542195,Valorant Champions 2025,Group Stage: \n\t\t\t\t\t\tOpening (A),2025-09-12 09:00:00,Paper Rex,Xi Lai Gaming,2 - 0,PatMen,13744.0,Paper Rex,...,7.0,-8.0,65%,89.0,35%,2.0,2.0,0.0,NaN,NaN


In [4]:
df.shape

(17792, 26)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17792 entries, 0 to 17791
Data columns (total 26 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   match_id       17792 non-null  int64  
 1   event_name     17792 non-null  str    
 2   event_stage    17792 non-null  str    
 3   match_date     17792 non-null  str    
 4   team1          17792 non-null  str    
 5   team2          17792 non-null  str    
 6   score_overall  17792 non-null  str    
 7   player_name    17792 non-null  str    
 8   player_id      17788 non-null  float64
 9   player_team    17792 non-null  str    
 10  stat_type      17792 non-null  str    
 11  agent          17772 non-null  str    
 12  rating         16382 non-null  float64
 13  acs            17702 non-null  float64
 14  k              17752 non-null  float64
 15  d              17752 non-null  float64
 16  a              17752 non-null  float64
 17  kd_diff        17752 non-null  float64
 18  kast           16

In [6]:
df.isna().sum()

match_id            0
event_name          0
event_stage         0
match_date          0
team1               0
team2               0
score_overall       0
player_name         0
player_id           4
player_team         0
stat_type           0
agent              20
rating           1410
acs                90
k                  40
d                  40
a                  40
kd_diff            40
kast             1320
adr              1390
hs_percent       1320
fk               1370
fd               1369
fk_fd_diff       1369
map_name         5032
map_winner       5032
dtype: int64

"20 rows are missing `agent`, all from Valorant Masters Bangkok's showmatch stage (Team International vs. Team Thailand). Checking other showmatch rows shows most do have complete stats — but as exhibition matches between non-competitive rosters, they don't reflect genuine competitive strategy, so all showmatch rows are excluded, not just the incomplete ones."

In [7]:
df = df[~df['event_stage'].str.contains('Showmatch', case=False, na=False)]

In [8]:
df.isna().sum()

match_id            0
event_name          0
event_stage         0
match_date          0
team1               0
team2               0
score_overall       0
player_name         0
player_id           0
player_team         0
stat_type           0
agent               0
rating           1350
acs                30
k                   0
d                   0
a                   0
kd_diff             0
kast             1260
adr              1330
hs_percent       1260
fk               1310
fd               1309
fk_fd_diff       1309
map_name         4972
map_winner       4972
dtype: int64

The raw files mix two row types per player: `overall` (match-wide aggregates across all maps, with a comma-separated agent list) and `map` (one row per player per map, single agent). Filtering to `stat_type == 'map'` keeps only the per-map granularity needed here, and incidentally resolves the remaining 4972 (after `Showmatch` removal) null `map_name`/`map_winner` values, which only occurred on `overall` rows.

In [9]:
df = df[df.stat_type == 'map']
df.isna().sum()

match_id           0
event_name         0
event_stage        0
match_date         0
team1              0
team2              0
score_overall      0
player_name        0
player_id          0
player_team        0
stat_type          0
agent              0
rating           990
acs               30
k                  0
d                  0
a                  0
kd_diff            0
kast             920
adr              970
hs_percent       920
fk               960
fd               959
fk_fd_diff       959
map_name           0
map_winner         0
dtype: int64

In [10]:
print(sorted(df.agent.unique()))
df.agent.nunique()

['Astra', 'Breach', 'Brimstone', 'Chamber', 'Clove', 'Cypher', 'Deadlock', 'Fade', 'Gekko', 'Harbor', 'Iso', 'Jett', 'Kayo', 'Killjoy', 'Neon', 'Omen', 'Phoenix', 'Raze', 'Reyna', 'Sage', 'Skye', 'Sova', 'Tejo', 'Viper', 'Vyse', 'Waylay', 'Yoru']


27

Valorant groups agents into four roles (Duelist, Initiator, Controller, Sentinel). Since team composition by role — not by specific agent — is the feature of interest, each agent is mapped to its role via a manually built reference dictionary, covering all 27 agents present in this dataset.

In [11]:
agent_to_role = {
    # Duelists
    'Iso' : 'Duelist', 'Jett' : 'Duelist', 'Neon' : 'Duelist', 'Phoenix' : 'Duelist', 'Raze' : 'Duelist', 'Reyna' : 'Duelist', 'Waylay' : 'Duelist', 'Yoru' : 'Duelist',
    # Initiators
    'Breach' : 'Initiator', 'Fade' : 'Initiator', 'Gekko' : 'Initiator', 'Kayo' : 'Initiator', 'Skye' : 'Initiator', 'Sova' : 'Initiator', 'Tejo' : 'Initiator',
    # Controllers
    'Astra' : 'Controller', 'Brimstone' : 'Controller', 'Clove' : 'Controller', 'Harbor' : 'Controller', 'Omen' : 'Controller', 'Viper' : 'Controller',
    # Sentinels
    'Chamber' : 'Sentinel', 'Cypher' : 'Sentinel', 'Deadlock' : 'Sentinel', 'Killjoy' : 'Sentinel', 'Sage' : 'Sentinel', 'Vyse' : 'Sentinel'
}

df['role'] = df['agent'].map(agent_to_role)
df.role.isna().sum()

np.int64(0)

In [12]:
df['won'] = (df.player_team == df.map_winner)

In [13]:
df = df[['match_id', 'player_team', 'agent', 'role', 'map_name', 'map_winner', 'won']]

In [14]:
df.head(10)

,match_id,player_team,agent,role,map_name,map_winner,won
10,542195,Paper Rex,Yoru,Duelist,Bind,Paper Rex,True
11,542195,Paper Rex,Raze,Duelist,Bind,Paper Rex,True
12,542195,Paper Rex,Brimstone,Controller,Bind,Paper Rex,True
13,542195,Paper Rex,Viper,Controller,Bind,Paper Rex,True
14,542195,Paper Rex,Fade,Initiator,Bind,Paper Rex,True
15,542195,Xi Lai Gaming,Raze,Duelist,Bind,Paper Rex,False
16,542195,Xi Lai Gaming,Brimstone,Controller,Bind,Paper Rex,False
17,542195,Xi Lai Gaming,Skye,Initiator,Bind,Paper Rex,False
18,542195,Xi Lai Gaming,Gekko,Initiator,Bind,Paper Rex,False
19,542195,Xi Lai Gaming,Viper,Controller,Bind,Paper Rex,False


For pick rate and win rate (by agent, and by agent per map), only `agent`, `map_name`, and `won` are needed — `won` is derived by comparing `player_team` against `map_winner`. This subset is saved as `agent_level.csv`.

In [15]:
df_agent_level = df[['agent', 'map_name','won']]
df_agent_level.to_csv('../data/processed/agent_level.csv', index=False)

Building team composition requires collapsing each team's 5 players (per map) into one row of role counts. Grouping by `match_id`, `player_team`, and `map_name` — rather than team and map alone — ensures a team's rematch on the same map later in the event is kept as a separate row, not merged with its earlier appearance.

In [16]:
comp_df = pd.crosstab([df.match_id, df.player_team, df.map_name], df.role).reset_index()
comp_df.head()

role,match_id,player_team,map_name,Controller,Duelist,Initiator,Sentinel
0,427991,Evil Geniuses,Pearl,1,2,1,1
1,427991,Evil Geniuses,Split,2,1,1,1
2,427991,LOUD,Pearl,1,1,2,1
3,427991,LOUD,Split,1,1,2,1
4,427992,100 Thieves,Bind,2,1,2,0


The crosstab above only carries forward the columns it was given, so `won` — needed as the model target — has to be merged back in separately, matched on the same three grouping keys. Since all 5 players on a team share the same `won` value, the source table is de-duplicated first to keep the merge one-to-one.

In [17]:
comp_df = comp_df.merge(
    df[['match_id', 'player_team', 'map_name', 'won']].drop_duplicates(),
    on=['match_id', 'player_team', 'map_name']
)
comp_df.head()

,match_id,player_team,map_name,Controller,Duelist,Initiator,Sentinel,won
0,427991,Evil Geniuses,Pearl,1,2,1,1,False
1,427991,Evil Geniuses,Split,2,1,1,1,False
2,427991,LOUD,Pearl,1,1,2,1,True
3,427991,LOUD,Split,1,1,2,1,True
4,427992,100 Thieves,Bind,2,1,2,0,True


Two versions are saved from the same composition table: `comp_only.csv` (role counts + outcome only) for a model trained without map context, and 
`comp_map.csv` (same, plus `map_name`) for a model trained with it — matching the two separate interactive predictors planned downstream.

In [18]:
df_comp = comp_df[['Controller', 'Duelist', 'Initiator', 'Sentinel', 'won']]
df_comp.to_csv('../data/processed/comp_only.csv', index=False)

In [19]:
df_comp_map = comp_df[['map_name', 'Controller', 'Duelist', 'Initiator', 'Sentinel', 'won']]
df_comp_map.to_csv('../data/processed/comp_map.csv', index=False)